In [10]:
import os
os.environ["HF_ENDPOINT"] = "https://hf-mirror.com"

import sys
sys.path.append("../../../../LoRO")

from utils import *

In [11]:
# Load model directly
from transformers import AutoTokenizer, AutoModelForSequenceClassification, pipeline
from datasets import load_dataset
import tqdm

tokenizer = AutoTokenizer.from_pretrained("valhalla/bart-large-sst2")
model = AutoModelForSequenceClassification.from_pretrained("valhalla/bart-large-sst2")

You passed along `num_labels=3` with an incompatible id to label map: {'0': 'NEGATIVE', '1': 'POSITIVE'}. The number of labels wil be overwritten to 2.


In [12]:
print(model)

BartForSequenceClassification(
  (model): BartModel(
    (shared): BartScaledWordEmbedding(50265, 1024, padding_idx=1)
    (encoder): BartEncoder(
      (embed_tokens): BartScaledWordEmbedding(50265, 1024, padding_idx=1)
      (embed_positions): BartLearnedPositionalEmbedding(1026, 1024)
      (layers): ModuleList(
        (0-11): 12 x BartEncoderLayer(
          (self_attn): BartAttention(
            (k_proj): Linear(in_features=1024, out_features=1024, bias=True)
            (v_proj): Linear(in_features=1024, out_features=1024, bias=True)
            (q_proj): Linear(in_features=1024, out_features=1024, bias=True)
            (out_proj): Linear(in_features=1024, out_features=1024, bias=True)
          )
          (self_attn_layer_norm): LayerNorm((1024,), eps=1e-05, elementwise_affine=True)
          (activation_fn): GELUActivation()
          (fc1): Linear(in_features=1024, out_features=4096, bias=True)
          (fc2): Linear(in_features=4096, out_features=1024, bias=True)
       

In [14]:
#obfuscate model
model = model_obfuscation(model)
print(model)

Obfuscating: model.encoder.layers.0.self_attn.k_proj
Obfuscating: model.encoder.layers.0.self_attn.v_proj
Obfuscating: model.encoder.layers.0.self_attn.q_proj
Obfuscating: model.encoder.layers.0.self_attn.out_proj
Obfuscating: model.encoder.layers.0.fc1
Obfuscating: model.encoder.layers.0.fc2
Obfuscating: model.encoder.layers.1.self_attn.k_proj
Obfuscating: model.encoder.layers.1.self_attn.v_proj
Obfuscating: model.encoder.layers.1.self_attn.q_proj
Obfuscating: model.encoder.layers.1.self_attn.out_proj
Obfuscating: model.encoder.layers.1.fc1
Obfuscating: model.encoder.layers.1.fc2
Obfuscating: model.encoder.layers.2.self_attn.k_proj
Obfuscating: model.encoder.layers.2.self_attn.v_proj
Obfuscating: model.encoder.layers.2.self_attn.q_proj
Obfuscating: model.encoder.layers.2.self_attn.out_proj
Obfuscating: model.encoder.layers.2.fc1
Obfuscating: model.encoder.layers.2.fc2
Obfuscating: model.encoder.layers.3.self_attn.k_proj
Obfuscating: model.encoder.layers.3.self_attn.v_proj
Obfuscating:

In [15]:
model = obfus_inference_mode(model)

In [16]:
classifier = pipeline("text-classification", model=model, tokenizer=tokenizer, device='cuda')

In [17]:
# two examples from training set

sentence = "contains no wit , only labored gags"

print(classifier(sentence))

sentence = "are more deeply thought through than in most ` right-thinking ' films"
print(classifier(sentence))

[{'label': 'NEGATIVE', 'score': 0.7391213178634644}]
[{'label': 'NEGATIVE', 'score': 0.7391191124916077}]


In [18]:
dataset = load_dataset("glue", "sst2")['validation']
print(dataset)

Dataset({
    features: ['sentence', 'label', 'idx'],
    num_rows: 872
})


In [19]:
correct = 0
total = 0

for i in tqdm.tqdm(range(872)):
    result = classifier(dataset[i]['sentence'])
    result = 0 if result[0]['label'] == 'NEGATIVE' else 1
    if result == dataset[i]['label']:
        correct += 1
    total += 1

print("correct:{}, total:{}, accuracy:{}".format(correct, total, correct/total))

100%|██████████| 872/872 [00:13<00:00, 66.16it/s]

correct:428, total:872, accuracy:0.4908256880733945
